In [1]:
%pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import shutil
from tqdm import tqdm

data_raw_dir = "../data/raw" 
processed_dir = "../data/processed/dataset_binario"

pasta_saudaveis = os.path.join(processed_dir, "Saudaveis")
pasta_leucemia = os.path.join(processed_dir, "Leucemia")
os.makedirs(pasta_saudaveis, exist_ok=True)
os.makedirs(pasta_leucemia, exist_ok=True)

# As pastas principais reais reveladas pelo ficheiro de checksum
pastas_leucemia = ['RUNX1_RUNX1T1', 'CBFB_MYH11', 'PML_RARA', 'NPM1']
pasta_controle = 'control'

print("⏳ A ler a estrutura genética das pastas...")

arquivos_para_copiar = []

for root, dirs, files in os.walk(data_raw_dir):
    for file in files:
        # Apenas ficheiros de imagem
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.tif')):
            src = os.path.join(root, file)
            
            # Normalizamos o caminho para não haver problemas entre barras do Windows (\) e Linux (/)
            caminho_partes = root.replace('\\', '/').split('/')
            
            destino = None
            # Verificamos em que "ramo" principal esta imagem está
            if pasta_controle in caminho_partes:
                destino = pasta_saudaveis
            else:
                for p_leucemia in pastas_leucemia:
                    if p_leucemia in caminho_partes:
                        destino = pasta_leucemia
                        break
                        
            if destino:
                # Extraímos as 3 letras do paciente para criar um nome de ficheiro único
                # Ex: KRG_image_276.tif
                pasta_paciente = os.path.basename(root) 
                nome_novo = f"{pasta_paciente}_{file}"
                dst = os.path.join(destino, nome_novo)
                
                arquivos_para_copiar.append((src, dst))

if len(arquivos_para_copiar) == 0:
    print("⚠️ Nenhuma imagem encontrada! Verifique se as imagens já estão descompactadas.")
else:
    imagens_copiadas = 0
    for src, dst in tqdm(arquivos_para_copiar, desc="A separar Leucemia vs Saudáveis"):
        shutil.copy2(src, dst)
        imagens_copiadas += 1

    print(f"\n✅ SUCESSO ABSOLUTO! {imagens_copiadas} imagens foram mapeadas e divididas perfeitamente para o PyTorch.")

⏳ A ler a estrutura genética das pastas...


A separar Leucemia vs Saudáveis: 100%|██████████| 81214/81214 [01:51<00:00, 729.48it/s] 


✅ SUCESSO ABSOLUTO! 81214 imagens foram mapeadas e divididas perfeitamente para o PyTorch.
